# AIDP hourly critical-event email monitor

Install **aidp_audit_alerts-0.4.0-py3-none-any.whl** on the monitoring cluster. Configure the recipient and Resend API key in Cell 1, and your AIDP connection/authentication/checkpoint path in Cell 2. Add this notebook to an AIDP job scheduled **every hour**, with **one active run**. Each invocation catches up, sends queued critical alerts, saves state and exits; there is no hour-long sleep loop.

- **First invocation:** read from current UTC time minus one hour through the invocation's fixed UTC cutoff.
- **Later invocations:** resume the saved read timestamp through the new cutoff. Missed schedules are caught up in bounded one-hour windows, not discarded.
- **Late logs:** re-read ten minutes before the saved cursor. Stable event-ID/per-recipient receipts suppress repeat emails from this overlap. Late ingestion beyond that overlap can still be missed; adjust from measured latency.
- **Delivery:** one email per critical event per recipient. Save the pending outbox before sending; save acceptance afterward. A rejection remains pending. An interrupted or uncertain send fails the job for operator reconciliation rather than blindly sending again.
- **No-event runs:** save the cursor and send no email.

Workspace state: `/path/to/persistent/workspace/aidp-alerts/hourly-state.json`. Keep that file and its `.lock` companion, and use the same mounted folder for every job run. This is separate from your manual deletion-test receipts, so the first production run considers its last hour afresh. Verify this workspace mount survives cluster replacement and supports file operations before relying on scheduling.

The source read checkpoint can advance before all emails are accepted because each pending delivery is already saved durably. Subsequent runs drain that queue. If the process cannot finish within its configured bounds, the job fails with saved progress rather than reporting success.

Cluster delete/edit and WORKFLOW DELETE/START/STOP/EXECUTE are CRITICAL. The workflow labels are based on your confirmed deployment mapping. All cluster/job names and initiating users visible to your OCI credentials are covered; failed attempts also alert with their recorded status. Separate source event IDs for separate START/EXECUTE/STOP actions are separate events.

An already accepted source event ID is not emailed again just because its status or payload changes. If the source provides no event ID, identical records use their full fingerprint; changed no-ID records can alert separately. Receipts are retained for 90 days; do not reset the state or backfill older history into this file. External email and filesystem writes cannot be one transaction, so uncertain outcomes require reconciliation; this is not a claim of exactly-once inbox delivery.

`onboarding@resend.dev` only supports your Resend account email. For team recipients, set FROM_ADDRESS to your verified-domain sender. Keys entered below are saved in notebook source: keep this notebook private; a managed RESEND_API_KEY environment variable is also supported without prompts.

No AIDP resources are deleted or edited by this notebook. Audit visibility is limited by the configured OCI identity. Schedule failures and stale runs need independent monitoring.

In [ ]:
# Cell 1 ? update these two values.
RECIPIENT_EMAIL = "REPLACE_WITH_YOUR_RESEND_ACCOUNT_EMAIL"
RESEND_API_KEY = "REPLACE_WITH_YOUR_RESEND_API_KEY"

In [ ]:
# Cell 2 ? configure your AIDP connection and persistent workspace checkpoint path.
ENDPOINT = 'https://aidp.REGION.oci.oraclecloud.com/20240831'
API_RESOURCE = "dataLakes"
PLATFORM_OCID = 'REPLACE_WITH_AIDP_PLATFORM_OCID'
OCI_CONFIG_FILE = "/path/to/oci_config"
OCI_PROFILE = "DEFAULT"
OCI_REGION = "REPLACE_WITH_OCI_REGION"
ENVIRONMENT = "PRODUCTION"
TIMEZONE = "Asia/Kolkata"
FROM_ADDRESS = "onboarding@resend.dev"  # use a verified-domain sender for other recipients
CHECKPOINT_PATH = "/path/to/persistent/workspace/aidp-alerts/hourly-state.json"
MAX_CYCLES_PER_RUN = 96

MONITOR = {
    "initial_lookback_minutes": 60,
    "window_minutes": 60,
    "ingestion_delay_seconds": 0,  # through this invocation's captured current time
    "overlap_seconds": 600,       # catch late arrivals; receipts prevent overlap resends
    "max_catchup_hours": 72,
    "poll_interval_seconds": 3600, # informational here; the external job owns scheduling
    "max_deliveries_per_cycle": 100,
    "max_pending_deliveries": 20000,
    "dedup_retention_days": 90,
    "dedup_mode": "event_id",
}

In [ ]:
# Cell 3 ? editable critical-event policy. No library changes needed for new rules.
RULES = [
    {"id": "cluster-delete", "enabled": True, "severity": "CRITICAL",
     "object_types": ["COMPUTE"], "operations": ["DELETE", "TERMINATE"], "statuses": ["*"]},
    {"id": "cluster-edit", "enabled": True, "severity": "CRITICAL",
     "object_types": ["COMPUTE"], "operations": ["UPDATE", "ATTACH", "DETACH", "RENAME"], "statuses": ["*"]},
    {"id": "job-delete", "enabled": True, "severity": "CRITICAL",
     "object_types": ["WORKFLOW"], "operations": ["DELETE"], "statuses": ["*"]},
    {"id": "job-run", "enabled": True, "severity": "CRITICAL",
     "object_types": ["WORKFLOW"], "operations": ["START", "STOP", "EXECUTE"], "statuses": ["*"]},
]
# To alert only on successful actions, change a rule's statuses to ["SUCCEEDED"].
# Policy additions retain the cursor; they apply to subsequent windows including overlap.
# Already queued messages keep their original content. Previously accepted IDs stay skipped.
# The monitor's own scheduled WORKFLOW run can also match the job-run rule.

In [ ]:
# Cell 4 ? validate and assemble configuration. No prompts; suitable for scheduled jobs.
import os
from pathlib import Path
from importlib.metadata import version

from aidp_alerts import validate_config
from aidp_alerts.runner import run_scheduled

# Existing deployments can supply the key as a managed environment variable instead.
_active_api_key = os.environ.get("RESEND_API_KEY") or RESEND_API_KEY
if "REPLACE_WITH" in RECIPIENT_EMAIL or not RECIPIENT_EMAIL.strip():
    raise ValueError("Update RECIPIENT_EMAIL in Cell 1.")
if "REPLACE_WITH" in _active_api_key or not _active_api_key.strip():
    raise ValueError("Update RESEND_API_KEY in Cell 1 or set its managed environment variable.")
if "REGION" in ENDPOINT or "REPLACE_WITH" in PLATFORM_OCID or "REPLACE_WITH" in OCI_REGION:
    raise ValueError("Configure ENDPOINT, PLATFORM_OCID and OCI_REGION in Cell 2.")
if CHECKPOINT_PATH.startswith("/path/to/"):
    raise ValueError("Configure a persistent kernel-readable CHECKPOINT_PATH in Cell 2.")
if not Path(OCI_CONFIG_FILE).is_file():
    raise ValueError("OCI config is not readable at the configured workspace path. Check/stage it for this notebook kernel.")

CONFIG = validate_config({
    "environment": ENVIRONMENT, "timezone": TIMEZONE,
    "aidp": {"endpoint": ENDPOINT, "api_resource": API_RESOURCE,
             "datalake_id": PLATFORM_OCID, "max_pages": 1000, "page_size": 100,
             "max_events": 100000, "max_attempts": 3, "request_timeout_seconds": 45},
    "oci": {"auth": "config_file", "config_file": OCI_CONFIG_FILE,
            "profile": OCI_PROFILE, "region": OCI_REGION},
    "monitor": MONITOR,
    "state": {"backend": "file", "path": CHECKPOINT_PATH,
              "durable_confirmed": True, "max_bytes": 52428800},
    "smtp": {"enabled": False},
    "resend": {"enabled": True, "from_address": FROM_ADDRESS,
               "recipients": [RECIPIENT_EMAIL], "api_key_ref": "resend_api_key",
               "timeout_seconds": 30, "min_interval_seconds": 0.25},
    "rules": RULES, "secret_refs": {}, "actor_aliases": {},
    "detail_fields": {}, "change_fields": [],
}, for_delivery=True)
# durable_confirmed=True selects your requested workspace storage. Verify mount
# persistence once before production scheduling; this flag does not prove persistence.

def secret_resolver(reference):
    if reference != "resend_api_key":
        raise ValueError("Unknown secret reference")
    return _active_api_key

print("Package version:", version("aidp-audit-alerts"))
print("Hourly critical-event policy validated. Checkpoint:", CHECKPOINT_PATH)

In [ ]:
# Cell 5 ? one scheduled invocation. Sends real email, then exits.
RESULT = run_scheduled(
    CONFIG,
    confirm_send=True,
    max_cycles=MAX_CYCLES_PER_RUN,
    secret_resolver=secret_resolver,
)
print("Hourly monitor completed.")
print("Source checkpoint UTC:", RESULT["cursor_utc"])
print("Run cutoff UTC:", RESULT["run_cutoff_utc"])
print("Email API requests acknowledged:", RESULT["email_accepted"])
if RESULT["email_accepted"] == 0:
    print("No new email was requested. Existing accepted events were skipped, or no critical events matched.")
print("See cycle logs for observed event types, matches, skipped receipts and pending delivery counts.")

## Scheduling and recovery

1. Install wheel 0.4.0 and fill Cell 1. Run the notebook once, then check the checkpoint file and actual inbox delivery.
2. Set the AIDP job to run this notebook every hour with one active run and automatic failure alerts. Do not run two copies against this file on different clusters.
3. Keep the same notebook path, mounted checkpoint folder, OCI config and library available to every invocation. Verify persistence across a restart using the same saved state.
4. Inspect the final `status=complete` summary and cycle metrics. A failed job must not be treated as successful just because its read cursor advanced: pending delivery records remain in the outbox.
5. On a saved `sending`/`uncertain` delivery, check Resend provider logs first. Use `from aidp_alerts.runner import resolve_delivery`, then `resolve_delivery(CONFIG, delivery_id, "accepted", confirm=True, secret_resolver=secret_resolver)` only with evidence of acceptance. `"retry"` explicitly accepts possible duplicates. Resend keys last 24 hours. Never delete the checkpoint or `.lock` file to recover.
6. If a gap exceeds 72 hours or the run exceeds its cycle/capacity limits, investigate source retention and increase bounds deliberately; saved progress is retained. Keep dedup receipt retention longer than the maximum catch-up and overlap.

This notebook is independently scheduled by your AIDP job; running it manually does not create or modify a schedule. Its mappings include your confirmed WORKFLOW operations. Deployment checks for job audit visibility, runtime paths, restart persistence, quotas and failure recovery remain necessary.